# Preprocess Ecoregions — Preprocessing Step 3

**Version:** 1.0 (2026-03-10) © R. Butzer

## What this notebook does
1. Loads EEA Biogeographic Regions 2016 shapefile
2. Reprojects and clips to the wildE study extent
3. Rasterises to 500m grid → `ecoregions_500m_3035_MBA.tif`
4. Exports attribute lookup tables with official EEA colours

## Inputs
- `_Runs\02_Woody_Cover\woody_cover_500m_3035.tif` — grid reference (from `02_reproject_woodycover.ipynb`)
- `thesis\data\Ecoregions\...\BiogeoRegions2016.shp` — EEA Biogeographic Regions 2016

## Outputs
- `_Runs\04_Ecoregions_MBA\ecoregions_500m_3035_MBA.tif` — rasterised ecoregion IDs
- `_Runs\04_Ecoregions_MBA\ecoregions_attributes.csv`
- `_Runs\04_Ecoregions_MBA\ecoregions_attributes_with_colors.csv` — with official EEA hex colours
  - → **Input for `05_analysis.ipynb`**

In [1]:
import rasterio
from rasterio.features import rasterize
import geopandas as gpd
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import os

In [2]:
# === KONFIGURATION ===

_workDir_remote = r"\\141.20.140.57\DAS_gsz1\_Biogeo\butzerfe\thesis"
_workDir_local  = r"D:\Seafile\Meine Bibliothek\uni\master\thesis"
workDir = _workDir_local


# Grid-Referenz (ersetzt combined_path)
woody_path = os.path.join(workDir, r"_Runs\02_Woody_Cover\woody_cover_500m_3035.tif")

# EEA Biogeographic Regions 2016
ecoregions_shp = os.path.join(workDir, r"_Primary_Data\Ecoregions\eea_v_3035_1_mio_biogeo-regions_p_2016_v01_r00\BiogeoRegions2016.shp")

# Outputs
output_dir = os.path.join(workDir, "_Runs", "03_Ecoregions_EEA")
os.makedirs(output_dir, exist_ok=True)

print("=" * 70)
print("ECOREGIONS PREPROCESSING")
print("=" * 70)
print(f"  Grid-Referenz: {woody_path}")
print(f"  EEA Shapefile: {ecoregions_shp}")
print(f"  Output-Ordner: {output_dir}")

ECOREGIONS PREPROCESSING
  Grid-Referenz: D:\Seafile\Meine Bibliothek\uni\master\thesis\_Runs\02_Woody_Cover\woody_cover_500m_3035.tif
  EEA Shapefile: D:\Seafile\Meine Bibliothek\uni\master\thesis\_Primary_Data\Ecoregions\eea_v_3035_1_mio_biogeo-regions_p_2016_v01_r00\BiogeoRegions2016.shp
  Output-Ordner: D:\Seafile\Meine Bibliothek\uni\master\thesis\_Runs\03_Ecoregions_EEA


In [4]:
# === Step 1: EEA Shapefile laden ===

print("\n1. LADE ECOREGIONS SHAPEFILE")
print("-" * 70)

ecoregions_gdf = gpd.read_file(ecoregions_shp)
print(f"✓ Shapefile geladen: {ecoregions_shp}")
print(f"  Anzahl Features: {len(ecoregions_gdf)}")
print(f"  Spalten: {list(ecoregions_gdf.columns)}")
print(f"  CRS: {ecoregions_gdf.crs}")
print(f"\nErste 5 Ecoregions:")
print(ecoregions_gdf.head())


1. LADE ECOREGIONS SHAPEFILE
----------------------------------------------------------------------
✓ Shapefile geladen: D:\Seafile\Meine Bibliothek\uni\master\thesis\_Primary_Data\Ecoregions\eea_v_3035_1_mio_biogeo-regions_p_2016_v01_r00\BiogeoRegions2016.shp
  Anzahl Features: 12
  Spalten: ['PK_UID', 'short_name', 'pre_2012', 'code', 'name', 'geometry']
  CRS: PROJCS["ETRS_1989_LAEA_L52_M10",GEOGCS["ETRS89",DATUM["European_Terrestrial_Reference_System_1989",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6258"]],PRIMEM["Greenwich",0],UNIT["Degree",0.0174532925199433]],PROJECTION["Lambert_Azimuthal_Equal_Area"],PARAMETER["latitude_of_center",52],PARAMETER["longitude_of_center",10],PARAMETER["false_easting",4321000],PARAMETER["false_northing",3210000],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH]]

Erste 5 Ecoregions:
   PK_UID short_name pre_2012       code                               name   
0       1   

In [5]:
# === Step 2: Raster-Eigenschaften aus woody_path lesen (Grid-Referenz) ===

print("\n2. LADE RASTER-EIGENSCHAFTEN (aus woody_cover_500m_3035.tif)")
print("-" * 70)

with rasterio.open(woody_path) as src:
    bounds    = src.bounds
    crs       = src.crs
    transform = src.transform
    width     = src.width
    height    = src.height

print(f"✓ Raster-Eigenschaften geladen: {woody_path}")
print(f"  Bounds (EPSG:3035): {bounds}")
print(f"  Dimensionen: {width} × {height}")
print(f"  Auflösung: {transform[0]}m")


2. LADE RASTER-EIGENSCHAFTEN (aus woody_cover_500m_3035.tif)
----------------------------------------------------------------------
✓ Raster-Eigenschaften geladen: D:\Seafile\Meine Bibliothek\uni\master\thesis\_Runs\02_Woody_Cover\woody_cover_500m_3035.tif
  Bounds (EPSG:3035): BoundingBox(left=2477500.0, bottom=1289000.0, right=7719000.0, top=6119000.0)
  Dimensionen: 10483 × 9660
  Auflösung: 500.0m


In [6]:
# === Step 3: Reprojektion und Clip auf Studiengebiet ===

print("\n3. TRANSFORMIERE ECOREGIONS")
print("-" * 70)

if ecoregions_gdf.crs != crs:
    print(f"Transformiere von {ecoregions_gdf.crs} zu {crs}")
    ecoregions_gdf = ecoregions_gdf.to_crs(crs)
else:
    print("✓ CRS stimmt bereits überein")

from shapely.geometry import box
study_bbox = box(*bounds)

print("Clippe Ecoregions auf Studiengebiet...")
ecoregions_clipped = gpd.clip(ecoregions_gdf, study_bbox)
print(f"✓ Anzahl Ecoregions im Studiengebiet: {len(ecoregions_clipped)}")
print(ecoregions_clipped[['short_name', 'pre_2012', 'code', 'name']])


3. TRANSFORMIERE ECOREGIONS
----------------------------------------------------------------------
Transformiere von PROJCS["ETRS_1989_LAEA_L52_M10",GEOGCS["ETRS89",DATUM["European_Terrestrial_Reference_System_1989",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6258"]],PRIMEM["Greenwich",0],UNIT["Degree",0.0174532925199433]],PROJECTION["Lambert_Azimuthal_Equal_Area"],PARAMETER["latitude_of_center",52],PARAMETER["longitude_of_center",10],PARAMETER["false_easting",4321000],PARAMETER["false_northing",3210000],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH]] zu EPSG:3035
Clippe Ecoregions auf Studiengebiet...
✓ Anzahl Ecoregions im Studiengebiet: 11
       short_name pre_2012           code   
1       anatolian      ANA      Anatolian  \
4        blackSea      BLS       BlackSea   
11        steppic      STE        Steppic   
6     continental      CON    Continental   
0          alpine      ALP         Alpine  

In [7]:
# === Step 4: Attribute auswählen, NUMERIC_ID erstellen, Rasterisierung ===

print("\n4. WÄHLE ATTRIBUTE")
print("-" * 70)

print(f"Verfügbare Spalten: {list(ecoregions_clipped.columns)}")

possible_id_cols   = ['code', 'CODE', 'region_id', 'REGION_ID', 'FID', 'OBJECTID']
possible_name_cols = ['name', 'NAME', 'short_name', 'ShortName', 'region_name', 'REGION_NAM']

id_column = next((c for c in possible_id_cols if c in ecoregions_clipped.columns), None)
name_column = next((c for c in possible_name_cols if c in ecoregions_clipped.columns), None)

if id_column is None:
    ecoregions_clipped['ECO_ID'] = range(1, len(ecoregions_clipped) + 1)
    id_column = 'ECO_ID'
    print(f"⚠ Keine ID-Spalte gefunden, erstelle eigene: {id_column}")
else:
    print(f"✓ Verwende ID-Spalte: {id_column}")

if name_column is None:
    ecoregions_clipped['ECO_NAME'] = [f"Region_{i}" for i in range(1, len(ecoregions_clipped) + 1)]
    name_column = 'ECO_NAME'
    print(f"⚠ Keine Name-Spalte gefunden, erstelle eigene: {name_column}")
else:
    print(f"✓ Verwende Name-Spalte: {name_column}")

if ecoregions_clipped[id_column].dtype == 'object':
    print(f"Konvertiere {id_column} zu numerischen Werten...")
    ecoregions_clipped['NUMERIC_ID'] = pd.factorize(ecoregions_clipped[id_column])[0] + 1
    numeric_id_col = 'NUMERIC_ID'
else:
    numeric_id_col = id_column

print(f"\nEcoregions im Studiengebiet:")
for _, row in ecoregions_clipped.iterrows():
    print(f"  ID {row[numeric_id_col]}: {row[name_column]}")

# === Step 5: Rasterisierung ===

print("\n5. RASTERISIERE ECOREGIONS")
print("-" * 70)

eco_raster = rasterize(
    [(geom, value) for geom, value in zip(
        ecoregions_clipped.geometry,
        ecoregions_clipped[numeric_id_col]
    )],
    out_shape=(height, width),
    transform=transform,
    fill=0,
    dtype='uint16'
)

print(f"✓ Raster erstellt: {eco_raster.shape}")
print(f"  Unique IDs: {np.unique(eco_raster[eco_raster > 0])}")

eco_raster_path = os.path.join(output_dir, "ecoregions_500m_3035_MBA.tif")
with rasterio.open(
    eco_raster_path, 'w',
    driver='GTiff', height=height, width=width,
    count=1, dtype='uint16', crs=crs, transform=transform,
    compress='lzw', nodata=0
) as dst:
    dst.write(eco_raster, 1)
    dst.set_band_description(1, 'Ecoregion_ID')

print(f"✓ Ecoregions-Raster gespeichert: {eco_raster_path}")

csv_path = os.path.join(output_dir, "ecoregions_attributes.csv")
ecoregions_clipped[[numeric_id_col, name_column]].to_csv(csv_path, index=False)
print(f"✓ Attribut-Tabelle gespeichert: {csv_path}")


4. WÄHLE ATTRIBUTE
----------------------------------------------------------------------
Verfügbare Spalten: ['PK_UID', 'short_name', 'pre_2012', 'code', 'name', 'geometry']
✓ Verwende ID-Spalte: code
✓ Verwende Name-Spalte: name
Konvertiere code zu numerischen Werten...

Ecoregions im Studiengebiet:
  ID 1: Anatolian Bio-geographical Region
  ID 2: Black Sea Bio-geographical Region
  ID 3: Steppic Bio-geographical Region
  ID 4: Continental Bio-geographical Region
  ID 5: Alpine Bio-geographical Region
  ID 6: Boreal Bio-geographical Region
  ID 7: Mediterranean Bio-geographical Region
  ID 8: Pannonian Bio-geographical Region
  ID 9: Atlantic Bio-geographical Region
  ID 10: outside Europe
  ID 11: Arctic Bio-geographical Region

5. RASTERISIERE ECOREGIONS
----------------------------------------------------------------------
✓ Raster erstellt: (9660, 10483)
  Unique IDs: [ 1  2  3  4  5  6  7  8  9 10 11]
✓ Ecoregions-Raster gespeichert: D:\Seafile\Meine Bibliothek\uni\master\thes

In [8]:
# === Step 6: Erweiterte Attribut-Tabelle mit EEA-Farben ===

print("\n6. SPEICHERE ATTRIBUT-TABELLE MIT EEA-FARBEN")
print("-" * 70)

# ECOREGION_COLORS = {
#     'ALP': '#e8a197',
#     'ANA': '#df826d',
#     'ARC': '#bbe4e7',
#     'ATL': '#0075a5',
#     'BLS': '#f3c6af',
#     'BOR': '#00668e',
#     'CON': '#738b54',
#     'MED': '#f7b916',
#     'PAN': '#9c6d48',
#     'STE': '#fedfac',
#     'OUT': '#dcddde'
# }


# abgewandelte farben zur besseren Unterscheidbarkeit
ECOREGION_COLORS = {
    'ALP': "#e89881",
    'ANA': '#d9562f',
    'ARC': '#7dd3d8',
    'ATL': '#0f88ca',
    'BLS': '#c56e05',
    'BOR': '#003d7a',
    'CON': '#5a7d3e',
    'MED': '#ffb700',
    'PAN': '#8b5a3c',
    'STE': '#fee090',
    'OUT': '#bfbfbf'
}

ecoregions_clipped['hex_color'] = ecoregions_clipped['pre_2012'].map(ECOREGION_COLORS)

missing_colors = ecoregions_clipped[ecoregions_clipped['hex_color'].isna()]['pre_2012'].unique()
if len(missing_colors) > 0:
    print(f"⚠️  Fehlende Farben für Codes: {missing_colors} → Standard-Grau (#808080)")
    ecoregions_clipped['hex_color'] = ecoregions_clipped['hex_color'].fillna('#808080')

def hex_to_rgb(hex_color):
    if pd.isna(hex_color):
        return (128, 128, 128)
    hex_color = hex_color.lstrip('#')
    return tuple(int(hex_color[i:i+2], 16) for i in (0, 2, 4))

ecoregions_clipped['rgb_r'] = ecoregions_clipped['hex_color'].apply(lambda x: hex_to_rgb(x)[0])
ecoregions_clipped['rgb_g'] = ecoregions_clipped['hex_color'].apply(lambda x: hex_to_rgb(x)[1])
ecoregions_clipped['rgb_b'] = ecoregions_clipped['hex_color'].apply(lambda x: hex_to_rgb(x)[2])

csv_extended = os.path.join(output_dir, "ecoregions_attributes_with_colors.csv")
ecoregions_clipped[[numeric_id_col, 'code', 'pre_2012', name_column, 'hex_color', 'rgb_r', 'rgb_g', 'rgb_b']].to_csv(
    csv_extended, index=False
)
print(f"✓ Farb-Attribut-Tabelle gespeichert: {csv_extended}")
print(ecoregions_clipped[['code', 'pre_2012', name_column, 'hex_color']].drop_duplicates().sort_values('pre_2012'))

print("\n" + "=" * 70)
print("✓✓✓ PREPROCESSING STEP 3 ABGESCHLOSSEN ✓✓✓")
print("=" * 70)
print(f"  Output (raster):       {eco_raster_path}")
print(f"  Output (attributes):   {csv_path}")
print(f"  Output (with colours): {csv_extended}")
print("  → Weiter mit: 04_export_modis_igbp.ipynb")
print("=" * 70)


6. SPEICHERE ATTRIBUT-TABELLE MIT EEA-FARBEN
----------------------------------------------------------------------
✓ Farb-Attribut-Tabelle gespeichert: D:\Seafile\Meine Bibliothek\uni\master\thesis\_Runs\03_Ecoregions_EEA\ecoregions_attributes_with_colors.csv
             code pre_2012                                   name hex_color
0          Alpine      ALP         Alpine Bio-geographical Region   #e89881
1       Anatolian      ANA      Anatolian Bio-geographical Region   #d9562f
2          Arctic      ARC         Arctic Bio-geographical Region   #7dd3d8
3        Atlantic      ATL       Atlantic Bio-geographical Region   #0f88ca
4        BlackSea      BLS      Black Sea Bio-geographical Region   #c56e05
5          Boreal      BOR         Boreal Bio-geographical Region   #003d7a
6     Continental      CON    Continental Bio-geographical Region   #5a7d3e
8   Mediterranean      MED  Mediterranean Bio-geographical Region   #ffb700
9         Outside      OUT                         out